In [1]:
!pip install --upgrade transformers

In [4]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# 1. SETTINGS
MODEL_CHECKPOINT = "t5-small" # Using 'small' for speed, switch to 'base' for quality
BATCH_SIZE = 16
NUM_EPOCHS = 3

# 2. DATA PREPARATION
# We use the PAWS dataset (Paraphrase Adversaries from Word Scrambling)
# We only want pairs where label=1 (meaning they ARE paraphrases)
print("Loading and filtering dataset...")
dataset = load_dataset("paws", "labeled_final")

def filter_paraphrases(example):
    return example['label'] == 1

paraphrase_dataset = dataset.filter(filter_paraphrases)

# Split into train/test (taking a smaller subset for assignment speed)
train_data = paraphrase_dataset['train'].select(range(2000)) # Train on 2000 samples
val_data = paraphrase_dataset['validation'].select(range(200))

# 3. TOKENIZATION
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def preprocess_function(examples):
     # 1. Prepare Inputs
    inputs = ["paraphrase: " + doc for doc in examples["sentence1"]]
    # FIX 1: Add padding="max_length" to ensure consistent shapes
    model_inputs = tokenizer(inputs, max_length=128, padding="max_length", truncation=True)

    # 2. Prepare Labels (Targets)
    labels = tokenizer(examples["sentence2"], max_length=128, padding="max_length", truncation=True)

    # Optional Optimization: Replace pad token ID with -100 so the model ignores padding in loss calculation
    # (This improves metric accuracy but isn't strictly required for it to run)
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_data.map(
    preprocess_function,
    batched=True,
    remove_columns=train_data.column_names # <--- CRITICAL FIX
)
tokenized_val = val_data.map(
    preprocess_function,
    batched=True,
    remove_columns=val_data.column_names   # <--- CRITICAL FIX
)

# 4. TRAINING SETUP
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)

args = Seq2SeqTrainingArguments(
    output_dir="./custom_paraphrase_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=NUM_EPOCHS,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(), # Use mixed precision if GPU is available
)


data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# 5. RUN TRAINING
print("Starting training...")
trainer.train()

# 6. SAVE MODEL
model.save_pretrained("./final_cpg_model")
tokenizer.save_pretrained("./final_cpg_model")
print("Model saved to ./final_cpg_model")

Loading and filtering dataset...


Filter:   0%|          | 0/49401 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

/tmp/ipython-input-3360285345.py:76: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Starting training...


Epoch,Training Loss,Validation Loss
1,No log,0.706800
2,No log,0.678092
3,No log,0.672719


Model saved to ./final_cpg_model


In [11]:
import torch
import nltk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from nltk.tokenize import sent_tokenize
import time

# Ensure NLTK data is ready
nltk.download('punkt')

# 1. SETUP: Use a model that KNOWS how to paraphrase
# We swap 't5-base' for a model pre-trained on the PAWS dataset
model_name = "Vamsi/T5_Paraphrase_Paws"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

# 2. THE CUSTOM LOGIC (Sentence-by-Sentence Pipeline)
def custom_paraphrase_pipeline(paragraph):
    # Split paragraph into sentences to preserve length
    sentences = sent_tokenize(paragraph)
    paraphrased_sentences = []

    print(f"Processing {len(sentences)} sentences...")

    for sentence in sentences:
        # The specific prefix this model expects
        text = "paraphrase: " + sentence + " </s>"

        encoding = tokenizer.encode_plus(
            text,
            padding="longest",
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"].to(device)
        attention_masks = encoding["attention_mask"].to(device)

        # Diverse Beam Search to encourage rewriting but keep length
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_masks,
            max_length=256,
            do_sample=False,  # Deterministic is usually safer for length
            num_beams=4,
            repetition_penalty=1.5, # Penalize repeating words
            early_stopping=True,
            num_return_sequences=1
        )

        output_str = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Fallback: If model fails or outputs empty, keep original sentence
        if len(output_str) < 5:
            paraphrased_sentences.append(sentence)
        else:
            paraphrased_sentences.append(output_str)

    # Rejoin sentences
    return " ".join(paraphrased_sentences)

# 3. TEST DATA (From Assignment)
input_text = """A cover letter is a formal document that accompanies your resume when you apply for a job. It serves as an introduction and provides additional context for your application. Here's a breakdown of its various aspects: Purpose The primary purpose of a cover letter is to introduce yourself to the hiring manager and to provide context for your resume. It allows you to elaborate on your qualifications, skills, and experiences in a way that your resume may not fully capture. It's also an opportunity to express your enthusiasm for the role and the company, and to explain why you would be a good fit. Content A typical cover letter includes the following sections: 1. Header: Includes your contact information, the date, and the employer's contact information. 2. Salutation: A greeting to the hiring manager, preferably personalized with their name. 3. Introduction: Briefly introduces who you are and the position you're applying for. 4. Body: This is the core of your cover letter where you discuss your qualifications, experiences, and skills that make you suitable for the job. You can also mention how you can contribute to the company. 5. Conclusion: Summarizes your points and reiterates your enthusiasm for the role. You can also include a call to action, like asking for an interview. 6. Signature: A polite closing ("Sincerely," "Best regards," etc.) followed by your name. Significance in the Job Application Process The cover letter is often the first document that a hiring manager will read, so it sets the tone for your entire application. It provides you with a chance to stand out among other applicants and to make a strong first impression. Some employers specifically require a cover letter, and failing to include one could result in your application being disregarded. In summary, a cover letter is an essential component of a job application that serves to introduce you, elaborate on your qualifications, and make a compelling case for why you should be considered for the position."""

# Clean input
input_text = input_text.replace('\n', ' ').strip()

# 4. EXECUTION
start_time = time.time()
generated_text = custom_paraphrase_pipeline(input_text)
latency = time.time() - start_time

# 5. METRICS
input_words = len(input_text.split())
output_words = len(generated_text.split())
ratio = (output_words / input_words) * 100

print("\n" + "="*40)
print("FINAL RESULTS")
print("="*40)
print(f"Input Length:  {input_words} words")
print(f"Output Length: {output_words} words")
print(f"Length Ratio:  {ratio:.2f}% (Goal: >80%)")
print(f"Latency:       {latency:.2f} seconds")
print("-" * 40)
print("GENERATED PARAGRAPH:")
print(generated_text)
print("="*40)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading Vamsi/T5_Paraphrase_Paws on cuda...


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Processing 24 sentences...

FINAL RESULTS
Input Length:  329 words
Output Length: 338 words
Length Ratio:  102.74% (Goal: >80%)
Latency:       9.77 seconds
----------------------------------------
GENERATED PARAGRAPH:
A cover letter is a formal document that accompanies your resume when you apply for a job . It serves as an introduction and provides additional context for your application . Here's a breakdown of its various aspects: Purpose The primary purpose of a cover letter is to introduce yourself to the hiring manager and provide context for your resume . It allows you to elaborate on your qualifications, skills and experiences in a way that your resume may not fully capture. It's also an opportunity to express your enthusiasm for the role and the company and explain why you would be a good fit . Content A typical cover letter includes the following sections: 1. Header: Includes your contact information, the date and the employer's contact information 2. Salutation: A greeting to